In [1]:
from dotenv import load_dotenv
import os
from supabase import create_client, Client
from pathlib import Path
import pandas as pd
from rich.progress import track, Progress
import time
from rich.console import Console
from tqdm.notebook import tqdm
import football_analytics.data.preprocessing as preprocessing
from importlib import reload
import json
import math
reload(preprocessing)

<module 'football_analytics.data.preprocessing' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\src\\football_analytics\\data\\preprocessing.py'>

## Loading Supabase Info

In [2]:
# Load variables from .env into the environment
load_dotenv()

# Read variables
supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_KEY")

In [3]:
# Initialize client
supabase: Client = create_client(supabase_url, supabase_key)

## Process Statsbomb Data

#### Shots

In [4]:
table_name = "shots"
response = supabase.table(table_name).select("*").limit(1).execute()
print(response)

data=[{'created_at': '2025-12-08T21:01:27.929943+00:00', 'statsbomb_event_id': '6ee144e8-8ce0-4dfd-9e75-9622a7c43e23', 'x1': 99.3, 'y1': 24.6, 'distance_to_goal': 25.800193797721754, 'angle_to_goal_deg': 14.300521019515893, 'keeper_distance': 21.005951537600005, 'min_defender_distance': 4.0804411526206295, 'avg_defender_distance': 9.960594456102717, 'num_def_in_shot_triangle': 1, 'num_teammates_in_box': 1, 'shot_to_min_def_ratio': 6.307435514918068, 'shot_type': 'Open Play', 'body_part': 'Right Foot', 'outcome': 'Off T', 'full_json': '{"id":"6ee144e8-8ce0-4dfd-9e75-9622a7c43e23","index":2417,"period":2,"timestamp":1765671206704,"minute":58,"second":26,"type":{"id":16,"name":"Shot"},"possession":135,"possession_team":{"id":227,"name":"Napoli"},"play_pattern":{"id":1,"name":"Regular Play"},"team":{"id":227,"name":"Napoli"},"duration":1.170437,"tactics":null,"related_events":["68fb446e-4186-4796-8141-50ed944dd718"],"off_camera":null,"player":{"id":7037,"name":"Lorenzo Insigne"},"position"

In [5]:
folder_path = Path("../../data/statsbomb/open-data-master/data/events")  # or absolute path: Path("/full/path/to/data")
json_files = list(folder_path.glob("*.json"))
shots_to_upsert = []
BATCHSIZE_SHOT_UPSERT = 50
MATCH_LIMIT = 1e6

# Loop over all json files
for match_index, json_file in tqdm(enumerate(json_files), desc="Processing...", total=len(json_files)):

    match_id = json_file.stem

    df = pd.read_json(json_file)
    df_shots = df[~df['shot'].isna()].copy().reset_index(drop=True)

    # Loop over shots
    for shot_index, row in df_shots.iterrows():
        event = row
        shot_data = preprocessing.extract_shot_features(event, match_id)
        shots_to_upsert.append(shot_data)

    if len(shots_to_upsert) >= BATCHSIZE_SHOT_UPSERT:
        response = supabase.table(table_name).upsert(shots_to_upsert, on_conflict="statsbomb_event_id").execute()
        shots_to_upsert = []

    if match_index > MATCH_LIMIT:
        break


if shots_to_upsert:
    response = supabase.table(table_name).upsert(shots_to_upsert, on_conflict="statsbomb_event_id").execute()
    shots_to_upsert = []


Processing...:   0%|          | 0/3464 [00:00<?, ?it/s]

#### Competion

In [19]:
json_file = "../../data/statsbomb/open-data-master/data/competitions.json"
with open(json_file, 'r') as f:
    competitions_to_upsert = json.load(f)

In [20]:
table_name = "competitions"
response = supabase.table(table_name).upsert(competitions_to_upsert, on_conflict="competition_id, season_id").execute()

#### Matches

#### Teams

In [22]:
table_name = "teams"
matches_folder = Path("../../data/statsbomb/open-data-master/data/matches")
match_files = list(matches_folder.glob("*/*.json"))
teams_by_id = {}

for match_file in tqdm(match_files, desc="Collecting teams", total=len(match_files)):
    matches = json.loads(match_file.read_bytes())
    for match in matches:
        for side in ("home_team", "away_team"):
            team_row = preprocessing.extract_team_row(match.get(side))
            if team_row and team_row.get("team_id"):
                teams_by_id[team_row["team_id"]] = team_row

teams_df = pd.DataFrame(teams_by_id.values())
# Replace NaN/NaT/inf with None to keep JSON serializable
teams_df = teams_df.replace([pd.NA, pd.NaT, float("inf"), float("-inf")], None)
teams_df = teams_df.where(pd.notnull(teams_df), None)
teams_to_upsert = []
for row in teams_df.to_dict(orient="records"):
    clean_row = {}
    for k, v in row.items():
        if pd.isna(v):
            clean_row[k] = None
            continue
        if isinstance(v, (float, int)) and math.isinf(v):
            clean_row[k] = None
            continue
        clean_row[k] = v
    teams_to_upsert.append(clean_row)



In [25]:
df_teams = pd.DataFrame(teams_to_upsert)

In [26]:
df_teams.head()

,team_id,team_name,team_gender,team_group,country_id,country_name,manager_id,manager_name,manager_nickname,manager_dob,manager_country_id,manager_country_name
0,217,Barcelona,male,None,214,Spain,NaN,None,None,None,NaN,None
1,207,Valencia,male,None,214,Spain,188.0,Javier Gracia Carlos,Javi Gracia,1970-05-01,214.0,Spain
2,219,RC Deportivo La Coruña,male,None,214,Spain,4519.0,Miguel Ángel Lotina Oruecheberría,Miguel Ángel Lotina,1957-06-18,214.0,Spain
3,220,Real Madrid,male,None,214,Spain,1003075.0,Alfredo Stéfano Di Stéfano Laulhé,Alfredo Di Stéfano,1926-07-04,11.0,Argentina
4,215,Athletic Club,male,None,214,Spain,NaN,None,None,None,NaN,None


In [27]:
response = supabase.table(table_name).upsert(teams_to_upsert, on_conflict="team_id").execute()
print(f"Prepared {len(teams_to_upsert)} teams for upsert")


Prepared 312 teams for upsert


#### Players

In [ ]:
table_name = "players"
import math
events_folder = Path("../../data/statsbomb/open-data-master/data/events")
event_files = list(events_folder.glob("*.json"))
players_by_id = {}

for event_file in tqdm(event_files, desc="Collecting players", total=len(event_files)):
    events = json.loads(event_file.read_bytes())
    for ev in events:
        player_row = preprocessing.extract_player_row(ev)
        if player_row and player_row.get("statsbomb_player_id"):
            players_by_id[player_row["statsbomb_player_id"]] = player_row

players_df = pd.DataFrame(players_by_id.values())
# Replace NaN/NaT/inf with None to keep JSON serializable
players_df = players_df.replace([pd.NA, pd.NaT, float("inf"), float("-inf")], None)
players_df = players_df.where(pd.notnull(players_df), None)



In [17]:
players_to_upsert = []
for row in players_df.to_dict(orient="records"):
    clean_row = {}
    for k, v in row.items():
        if pd.isna(v):
            clean_row[k] = None
            continue
        if isinstance(v, (float, int)) and math.isinf(v):
            clean_row[k] = None
            continue
        clean_row[k] = v
    players_to_upsert.append(clean_row)



In [19]:
if players_to_upsert:
    response = supabase.table(table_name).upsert(players_to_upsert, on_conflict="statsbomb_player_id").execute()
else:
    print("No players collected")
print(f"Prepared {len(players_to_upsert)} players for upsert")


Prepared 9043 players for upsert
